In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.config import PROCESSED_DIR, REPORTS_DIR, WHO_PM25_ANNUAL, WHO_PM25_24H

# Ensure output dir exists
(REPORTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

# Load master
df = pd.read_parquet(PROCESSED_DIR / 'jakarta_master.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

print(f'Loaded: {df.shape}')
print(f'PM2.5 valid: {df["pm25_mean"].notna().sum():,} days')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')

Loaded: (1826, 26)
PM2.5 valid: 1,355 days
Date range: 2019-01-01 to 2023-12-31


In [2]:
# Annual statistics
annual = df.groupby('year')['pm25_mean'].agg([
    'mean', 'median', 'std',
    lambda x: x.quantile(0.25),
    lambda x: x.quantile(0.75),
    lambda x: (x > WHO_PM25_24H).mean() * 100,
]).round(2)

annual.columns = ['mean', 'median', 'std', 'q25', 'q75', 'pct_exceed_who']
annual['who_annual_guideline'] = WHO_PM25_ANNUAL

print('=== Annual PM2.5 Summary (Jakarta Central) ===')
print(annual)
print(f'\nWHO Annual Guideline: {WHO_PM25_ANNUAL} µg/m³')
print(f'Jakarta average is {annual["mean"].mean()/WHO_PM25_ANNUAL:.1f}x above WHO guideline')

=== Annual PM2.5 Summary (Jakarta Central) ===
       mean  median    std    q25    q75  pct_exceed_who  who_annual_guideline
year                                                                          
2019  43.22   41.17  33.13  29.98  49.89           84.38                   5.0
2020  35.09   32.79  19.93  22.50  45.21           71.04                   5.0
2021  39.62   34.04  46.99  22.23  43.08           73.97                   5.0
2022  32.89   27.00  36.12  19.98  35.60           61.10                   5.0
2023  46.08   44.64  25.29  34.71  53.50           45.21                   5.0

WHO Annual Guideline: 5.0 µg/m³
Jakarta average is 7.9x above WHO guideline


In [3]:
import pymannkendall as mk
from scipy import stats

# Annual means untuk trend test
annual_means = df.groupby('year')['pm25_mean'].mean().dropna()
monthly_means = df.groupby(['year', 'month'])['pm25_mean'].mean().dropna()

# Test autocorrelation (Ljung-Box)
import statsmodels.stats.diagnostic as smd
lb_result = smd.acorr_ljungbox(
    df['pm25_mean'].dropna(),
    lags=[10],
    return_df=True
)
has_autocorr = lb_result['lb_pvalue'].iloc[0] < 0.05
print(f'Autocorrelation detected: {has_autocorr}')
print(f'Ljung-Box p-value: {lb_result["lb_pvalue"].iloc[0]:.4f}')

# Choose appropriate MK test
if has_autocorr:
    result = mk.hamed_rao_modification_test(monthly_means.values)
    method = 'Hamed-Rao Modified Mann-Kendall'
else:
    result = mk.original_test(monthly_means.values)
    method = 'Original Mann-Kendall'

print(f'\n=== Trend Test: {method} ===')
print(f'Trend      : {result.trend}')
print(f'p-value    : {result.p:.4f}')
print(f'Tau        : {result.Tau:.4f}')
print(f"Sen's slope: {result.slope * 12:.4f} µg/m³ per year")
print(f"\nConclusion : {'Significant trend detected' if result.p < 0.05 else 'No significant trend'} (α=0.05)")

Autocorrelation detected: True
Ljung-Box p-value: 0.0000

=== Trend Test: Hamed-Rao Modified Mann-Kendall ===
Trend      : no trend
p-value    : 0.9123
Tau        : 0.0103
Sen's slope: 0.2117 µg/m³ per year

Conclusion : No significant trend (α=0.05)


In [4]:
# Rolling 30-day mean
df['pm25_rolling30'] = df['pm25_mean'].rolling(30, min_periods=15).mean()

# Annual mean per year
annual_plot = df.groupby('year')['pm25_mean'].mean().reset_index()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Daily PM2.5 with 30-Day Rolling Mean', 'Annual Mean PM2.5'),
    row_heights=[0.65, 0.35],
    vertical_spacing=0.12
)

# Daily + rolling
fig.add_trace(go.Scatter(
    x=df.index, y=df['pm25_mean'],
    mode='lines', name='Daily PM2.5',
    line=dict(color='lightblue', width=0.8),
    opacity=0.6,
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['pm25_rolling30'],
    mode='lines', name='30-Day Rolling Mean',
    line=dict(color='steelblue', width=2),
), row=1, col=1)

# WHO guideline — use add_shape for subplot compatibility
fig.add_shape(
    type='line',
    x0=df.index.min(), x1=df.index.max(),
    y0=WHO_PM25_24H, y1=WHO_PM25_24H,
    line=dict(color='red', width=1.5, dash='dash'),
    row=1, col=1
)
fig.add_annotation(
    x=df.index.max(), y=WHO_PM25_24H,
    text='WHO 24h (15 µg/m³)', showarrow=False,
    xanchor='right', yanchor='bottom',
    font=dict(color='red', size=11),
    row=1, col=1
)

# PSBB shading
psbb_periods = [
    ('2020-04-10', '2020-06-04', 'PSBB Strict'),
    ('2020-06-05', '2020-09-13', 'PSBB Transition'),
    ('2020-09-14', '2020-10-11', 'PSBB Strict 2'),
]
colors = ['rgba(255,100,100,0.15)', 'rgba(255,165,0,0.15)', 'rgba(255,100,100,0.15)']

for (start, end, label), color in zip(psbb_periods, colors):
    fig.add_vrect(
        x0=start, x1=end,
        fillcolor=color, opacity=1,
        layer='below', line_width=0,
        annotation_text=label, annotation_position='top left',
        row=1, col=1
    )

# Annual bar
fig.add_trace(go.Bar(
    x=annual_plot['year'], y=annual_plot['pm25_mean'],
    name='Annual Mean',
    marker_color='steelblue',
    text=annual_plot['pm25_mean'].round(1),
    textposition='outside',
), row=2, col=1)

fig.add_shape(
    type='line',
    x0=annual_plot['year'].min() - 0.5,
    x1=annual_plot['year'].max() + 0.5,
    y0=WHO_PM25_ANNUAL, y1=WHO_PM25_ANNUAL,
    line=dict(color='red', width=1.5, dash='dash'),
    row=2, col=1
)
fig.add_annotation(
    x=annual_plot['year'].max() + 0.5, y=WHO_PM25_ANNUAL,
    text='WHO Annual (5 µg/m³)', showarrow=False,
    xanchor='right', yanchor='bottom',
    font=dict(color='red', size=11),
    row=2, col=1
)

fig.update_layout(
    title='Jakarta PM2.5 Air Quality 2019-2023',
    height=700,
    showlegend=True,
    template='plotly_white',
)
fig.update_yaxes(title_text='PM2.5 (µg/m³)', row=1, col=1)
fig.update_yaxes(title_text='PM2.5 (µg/m³)', row=2, col=1)

fig.show()

out = str(REPORTS_DIR / 'figures' / '01_temporal_trend.html')
fig.write_html(out)
print(f'Figure saved: {out}')

Figure saved: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\reports\figures\01_temporal_trend.html


In [5]:
# Monthly boxplot
monthly_df = df.reset_index()
monthly_df['month_name'] = monthly_df['date'].dt.strftime('%b')
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

fig2 = px.box(
    monthly_df.dropna(subset=['pm25_mean']),
    x='month_name', y='pm25_mean',
    category_orders={'month_name': month_order},
    color='season',
    color_discrete_map={'wet': 'steelblue', 'dry': 'darkorange'},
    title='Jakarta PM2.5 Monthly Distribution (2019-2023)',
    labels={'pm25_mean': 'PM2.5 (µg/m³)', 'month_name': 'Month'},
    template='plotly_white',
)

fig2.add_hline(
    y=WHO_PM25_24H, line_dash='dash',
    line_color='red',
    annotation_text='WHO 24h guideline'
)

fig2.update_layout(height=500)
fig2.show()

out2 = str(REPORTS_DIR / 'figures' / '02_seasonal_pattern.html')
fig2.write_html(out2)
print(f'Figure saved: {out2}')

Figure saved: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\reports\figures\02_seasonal_pattern.html


In [6]:
# Pivot table
pivot = df.groupby(['year', 'month'])['pm25_mean'].mean().unstack()
pivot.columns = month_order

fig3 = px.imshow(
    pivot,
    color_continuous_scale='RdYlGn_r',
    title='Jakarta PM2.5 Heatmap (Year x Month)',
    labels=dict(x='Month', y='Year', color='PM2.5 (µg/m³)'),
    text_auto='.0f',
    aspect='auto',
    template='plotly_white',
)

fig3.update_layout(height=400)
fig3.show()

out3 = str(REPORTS_DIR / 'figures' / '03_heatmap.html')
fig3.write_html(out3)
print(f'Figure saved: {out3}')

Figure saved: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\reports\figures\03_heatmap.html


In [7]:
# Investigate 2020 Oktober anomaly
oct_2020 = df[(df.index.year == 2020) & (df.index.month == 10)]
print("=== Oktober 2020 Detail ===")
print(oct_2020[["pm25_mean", "coverage_pct", "obs_count", "precipitation", "windspeed_max"]])
print(f"\nMean PM2.5 Oct 2020: {oct_2020['pm25_mean'].mean():.1f} µg/m³")
print(f"Coverage mean: {oct_2020['coverage_pct'].mean():.1f}%")

=== Oktober 2020 Detail ===
            pm25_mean  coverage_pct  obs_count  precipitation  windspeed_max
date                                                                        
2020-10-01        NaN         100.0       24.0           0.82           5.57
2020-10-02        NaN         100.0       24.0           2.47           4.07
2020-10-03        NaN         100.0       24.0          12.89           2.97
2020-10-04        NaN          33.0        8.0           5.54           4.02
2020-10-05        NaN          67.0       16.0           2.40           3.64
2020-10-06        NaN         100.0       24.0           0.85           3.66
2020-10-07        NaN         100.0       24.0           0.46           4.01
2020-10-08        NaN         100.0       24.0           2.05           3.55
2020-10-09        NaN         100.0       24.0           2.98           4.41
2020-10-10        NaN         100.0       24.0           3.68           2.88
2020-10-11        NaN         100.0       24.0  

In [8]:
# Document known data gaps
gaps = df[df["pm25_mean"].isna()].copy()
gaps_by_year = gaps.groupby("year").size()

print("=== Known Data Gaps by Year ===")
print(gaps_by_year)
print(f"\nTotal missing days: {len(gaps)}")
print(f"\nLargest consecutive gap:")

# Find longest consecutive NaN streak
is_nan = df["pm25_mean"].isna()
streak = 0
max_streak = 0
max_streak_start = None
current_start = None

for date, val in is_nan.items():
    if val:
        if streak == 0:
            current_start = date
        streak += 1
        if streak > max_streak:
            max_streak = streak
            max_streak_start = current_start
    else:
        streak = 0

print(f"  Start: {max_streak_start.date()}")
print(f"  Length: {max_streak} days")
print(f"\nAll gaps confirmed — no imputation needed for ITS analysis")
print(f"   (statsmodels handles NaN natively)")

=== Known Data Gaps by Year ===
year
2019     38
2020     73
2021     66
2022    102
2023    192
dtype: int64

Total missing days: 471

Largest consecutive gap:
  Start: 2022-10-24
  Length: 85 days

All gaps confirmed — no imputation needed for ITS analysis
   (statsmodels handles NaN natively)


In [9]:
# Cell 9 — Revised scope decision
print("=== REVISED ANALYSIS SCOPE ===")

# Check 2023 coverage lebih detail
df_2023 = df[df.index.year == 2023]
valid_2023 = df_2023["pm25_mean"].notna().sum()
print(f"2023 valid days: {valid_2023}/365 ({valid_2023/365*100:.1f}%)")

# Check per-year coverage
for year in range(2019, 2024):
    yr = df[df.index.year == year]
    valid = yr["pm25_mean"].notna().sum()
    total = len(yr)
    pct = valid/total*100
    flag = "OK" if pct >= 70 else "WARNING" if pct >= 50 else "INSUFFICIENT"
    print(f"{year}: {valid}/{total} days ({pct:.1f}%) [{flag}]")

print(f"\nDecision: Use 2019-2022 for trend analysis")
print(f"         2023 retained only for seasonal comparison with caveat")
print(f"         COVID analysis: 2019-2021 (sufficient coverage)")

=== REVISED ANALYSIS SCOPE ===
2023 valid days: 173/365 (47.4%)
2019: 327/365 days (89.6%) [OK]
2020: 293/366 days (80.1%) [OK]
2021: 299/365 days (81.9%) [OK]
2022: 263/365 days (72.1%) [OK]
2023: 173/365 days (47.4%) [INSUFFICIENT]

Decision: Use 2019-2022 for trend analysis
         2023 retained only for seasonal comparison with caveat
         COVID analysis: 2019-2021 (sufficient coverage)
